# Seed ToonCrafter weights to Kaggle dataset

Downloads `model.ckpt` (~10.5GB) from HuggingFace and publishes it as
`marsfmmg/tooncrafter-weights`. Run once. Future notebooks attach the dataset instead
of re-downloading.


In [ ]:
# === Download model.ckpt from HuggingFace ===
import subprocess, os

SEED = "/kaggle/working/seed"
os.makedirs(SEED, exist_ok=True)
URL = "https://huggingface.co/Doubiiu/ToonCrafter/resolve/main/model.ckpt"
EXPECTED = 10502492136
OUT_PATH = os.path.join(SEED, "model.ckpt")

if os.path.exists(OUT_PATH) and os.path.getsize(OUT_PATH) == EXPECTED:
    print(f"model.ckpt already present ({os.path.getsize(OUT_PATH)/1e9:.1f} GB) — skipping download")
else:
    print(f"Downloading model.ckpt ({EXPECTED/1e9:.1f} GB) from HuggingFace...")
    subprocess.run(["wget", "-q", "--show-progress", URL, "-O", OUT_PATH], check=True)
    actual = os.path.getsize(OUT_PATH)
    print(f"Downloaded: {actual/1e9:.2f} GB")
    assert actual == EXPECTED, f"SIZE MISMATCH: got {actual}, expected {EXPECTED}"
    print("Size verified.")


In [ ]:
# === Publish as dataset version ===
import subprocess, json, os

SEED = "/kaggle/working/seed"
DATASET_ID = "marsfmmg/tooncrafter-weights"

# Write dataset metadata (required for `kaggle datasets version`)
meta = {
    "title": "ToonCrafter Model Weights",
    "id": DATASET_ID,
    "licenses": [{"name": "Apache-2.0"}],
}
with open(os.path.join(SEED, "dataset-metadata.json"), "w") as f:
    json.dump(meta, f, indent=2)
print("Metadata written.")

# Publish (kernel is pre-authenticated as notebook owner)
print("Publishing dataset version (this takes a while for 10GB)...")
r = subprocess.run(
    ["kaggle", "datasets", "version", "-p", SEED,
     "-m", "ToonCrafter model.ckpt 10.5GB — seeded from HuggingFace"],
    capture_output=True, text=True, timeout=3600)
print(r.stdout[-500:] if r.stdout else "")
print(r.stderr[-500:] if r.stderr else "")
assert r.returncode == 0, "Dataset publish FAILED"
print("\nDone. Weights are now cached at", DATASET_ID)
